<a href="https://colab.research.google.com/github/techasit239/Final-Project---DADS6003/blob/main/Features_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from statistics import pstdev
from typing import List, Tuple
from pathlib import Path

!pip install pandas numpy nltk textstat
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
import pandas as pd
import numpy as np
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag
from textstat import textstat # สำหรับการนับพยางค์และความซับซ้อนของคำ
from google.colab import files

# =========================
# LOADERS
# =========================
def load_email_excel(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_excel(p, engine="openpyxl")
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def load_email_csv(path: str, encoding: str = "utf-8") -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_csv(p, encoding=encoding)
    _validate_cols(df)
    _sanitize_cols(df)
    return df

def _validate_cols(df: pd.DataFrame):
    required = {"Subject", "Body", "Label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"คอลัมน์หายไป: {missing} (ต้องมี {required})")

def _sanitize_cols(df: pd.DataFrame):
    df["Subject"] = df["Subject"].astype(str).fillna("")
    df["Body"]    = df["Body"].astype(str).fillna("")
    # ตามต้องการ: แปลง Label เป็น int ก็ได้ (คอมเมนต์บรรทัดถัดไปถ้า label เป็น string)
    # df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(-1).astype(int)

# =========================
# FEATURE EXTRACTORS
# (ครบตามรูป: Complexity + Stylistic)
# =========================
POLITENESS = ["please", "thank", "appreciate", "thanks", "appreciated", "appreciates", "appreciation"]
AGGRESSIVE = ["must", "now", "immediately"]
URGENCY    = ["urgent", "asap", "immediately"]
CONDITIONAL = ["if", "unless"]
PERSONAL_TAGS = ["[recipient’s name]", "[recipient's name],[Your Name],[Your name]"]   # รองรับ ’ และ '
WORD_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")

def strip_urls_emails(text: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)     # ตัด URL
    text = re.sub(r"\S+@\S+\.\S+", " ", text)              # ตัดอีเมล
    return text

def tokenize_words(text: str) -> List[str]:
    text = text if isinstance(text, str) else ""
    text = strip_urls_emails(text)
    return [m.group(0).lower() for m in WORD_RE.finditer(text)]

def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def count_markers(tokens: List[str], vocab: List[str]) -> int:
    vocab_set = set(w.lower() for w in vocab)
    return sum(1 for t in tokens if t in vocab_set)

def count_phrase_occurrences(text: str, phrases: List[str]) -> int:
    lowered = (text or "").lower()
    return sum(len(re.findall(re.escape(p.lower()), lowered)) for p in phrases)

def extract_stylo_features(email_text: str) -> dict:
    tokens = tokenize_words(email_text)

    # Complexity
    bigrams  = make_ngrams(tokens, 2)
    trigrams = make_ngrams(tokens, 3)
    word_lengths = [len(w) for w in tokens] or [0]
    word_len_var = pstdev(word_lengths)  # ส่วนเบี่ยงเบนมาตรฐาน (population)

    # Stylistic
    politeness_cnt   = count_markers(tokens, POLITENESS)
    aggressive_cnt   = count_markers(tokens, AGGRESSIVE)
    urgency_cnt      = count_markers(tokens, URGENCY)
    conditional_cnt  = count_markers(tokens, CONDITIONAL)
    personal_token_cnt = count_markers(tokens, ["you", "your"])
    personal_tag_cnt   = count_phrase_occurrences(email_text, PERSONAL_TAGS)
    personalisation_cnt = personal_token_cnt + personal_tag_cnt

    return {
        # Complexity
        "bigram_total_count": len(bigrams),
        "bigram_unique_count": len(set(bigrams)),
        "trigram_total_count": len(trigrams),
        "trigram_unique_count": len(set(trigrams)),
        "word_length_variation_std": float(word_len_var),
        # Stylistic
        "politeness_markers_count": politeness_cnt,
        "aggressiveness_markers_count": aggressive_cnt,
        "urgency_markers_count": urgency_cnt,
        "conditional_phrases_count": conditional_cnt,
        "personalisation_markers_count": personalisation_cnt,
    }





# กำหนดรายชื่อ Function Words, Prepositions, และ Pronouns
FUNCTION_WORDS = {'the', 'is', 'at', 'which', 'on', 'and', 'or', 'but', 'because'}
PREPOSITIONS = {'in', 'on', 'at', 'by', 'with'}
PRONOUN_TAGS = {'PRP', 'PRP$', 'WP', 'WP$'} # แท็ก NLTK สำหรับคำสรรพนาม (Personal, Possessive, Wh-pronouns)
LINKING_WORDS = {'but', 'and', 'or', 'because'} # คำที่ใช้ในการคำนวณ Sentence Complexity

def analyze_email_body(text):
    """
    คำนวณคุณลักษณะทางภาษาทั้งหมดจากข้อความอีเมล.
    """
    if pd.isna(text) or text is None:
        return (0,) * 19 # คืนค่า 0 ทั้งหมดถ้าข้อความเป็นค่าว่าง

    # 1. การเตรียมการ (Tokenization)
    text = str(text).lower() # แปลงเป็นตัวพิมพ์เล็กสำหรับวิเคราะห์คำส่วนใหญ่
    original_text = str(text) # เก็บข้อความดั้งเดิมไว้สำหรับบางการนับ

    # ใช้วิธี tokenize ของ NLTK สำหรับคำและประโยค
    words = word_tokenize(original_text)
    lower_words = [word.lower() for word in words if word.isalnum()] # กรองเอาเฉพาะคำที่เป็นตัวอักษร/ตัวเลข
    sentences = sent_tokenize(original_text)

    # คำนวณเบื้องต้น
    word_count = len(lower_words)
    char_count = len(original_text)
    sentence_count = len(sentences)

    # จัดการกรณีที่ word_count เป็น 0 เพื่อป้องกันการหารด้วยศูนย์
    if word_count == 0:
        return (0,) * 19


    # --- 1-7. Word Counts, Lengths, and Diversity ---

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in lower_words) / word_count

    # 5. Average Sentence Length
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0

    # 6. Unique Word Count
    unique_word_count = len(set(lower_words))

    # 7. Lexical Diversity
    lexical_diversity = unique_word_count / word_count


    # --- 8-10. Email, Uppercase Counts ---

    # 8. Number of Emails (นับคำที่มี @)
    email_count = sum(1 for word in words if '@' in word)

    # 9. Uppercase Word Count (นับคำที่เป็นตัวพิมพ์ใหญ่ทั้งหมดในคำเดิม)
    # ต้องใช้ original_text และ words ที่ยังไม่ได้แปลงเป็นตัวเล็ก
    uppercase_words = word_tokenize(str(text)) # Tokenize อีกครั้งเพื่อใช้คำเดิม
    upper_count = sum(1 for word in uppercase_words if word.isupper() and word.isalpha())

    # 10. Uppercase Word Count Ratio
    upper_ratio = upper_count / word_count


    # --- 11-12. Complexity Measures ---

    # 11. Complex Words Count (คำที่มีตัวอักษร > 6 ตัว)
    complex_count = sum(1 for word in lower_words if len(word) > 6)

    # 12. Average Syllables per Word (ใช้ textstat)
    try:
        total_syllables = sum(textstat.syllable_count(word) for word in lower_words)
        avg_syllables = total_syllables / word_count
    except:
        avg_syllables = 0


    # --- 13-14. Punctuation Counts ---

    # 13. Comma, Semicolon, and Colon Counts
    comma_count = original_text.count(',')
    semicolon_count = original_text.count(';')
    colon_count = original_text.count(':')

    # 14. Exclamation Count, Quotation Count, Dash Count
    exclamation_count = original_text.count('!')
    quotation_count = original_text.count('"')
    dash_count = original_text.count('-')


    # --- 15-19. Density and Ratio Measures ---

    # 15. Sentence Complexity Ratio (สัดส่วนของ but, and, or, because เทียบกับจำนวนคำทั้งหมด)
    linking_word_count = sum(1 for word in lower_words if word in LINKING_WORDS)
    sentence_complexity_ratio = linking_word_count / word_count

    # 16. Clause Density (สัดส่วนของ linking_word_count / sentence_count)
    clause_density = linking_word_count / sentence_count if sentence_count > 0 else 0

    # 17. Pronoun Density (NLTK POS Tagging)
    tagged_words = pos_tag(words) # ใช้คำที่ไม่ได้แปลงเป็นตัวเล็กในการ Tag
    pronoun_count = sum(1 for word, tag in tagged_words if tag in PRONOUN_TAGS)
    pronoun_density = pronoun_count / word_count

    # 18. Preposition Density
    preposition_count = sum(1 for word in lower_words if word in PREPOSITIONS)
    preposition_density = preposition_count / word_count

    # 19. Function Word Density
    function_word_count = sum(1 for word in lower_words if word in FUNCTION_WORDS)
    function_word_density = function_word_count / word_count


    # Return all 23 values
    return (
        word_count, char_count, avg_word_length, sentence_count, avg_sentence_length,
        unique_word_count, lexical_diversity, email_count, upper_count, upper_ratio,
        complex_count, avg_syllables,
        comma_count, semicolon_count, colon_count, exclamation_count, quotation_count, dash_count,
        sentence_complexity_ratio, clause_density, pronoun_density, preposition_density, function_word_density
    )

# กำหนดรายชื่อคอลัมน์ใหม่ตามลำดับที่ส่งคืนในฟังก์ชัน
NEW_COLUMNS = [
    'Word_Count', 'Character_Count', 'Average_Word_Length', 'Sentence_Count', 'Average_Sentence_Length',
    'Unique_Word_Count', 'Lexical_Diversity', 'Email_Count', 'Uppercase_Word_Count', 'Uppercase_Word_Count_Ratio',
    'Complex_Words_Count', 'Average_Syllables_per_Word',
    'Comma_Count', 'Semicolon_Count', 'Colon_Count', 'Exclamation_Count', 'Quotation_Count', 'Dash_Count',
    'Sentence_Complexity_Ratio', 'Clause_Density', 'Pronoun_Density', 'Preposition_Density', 'Function_Word_Density'
]
# =========================
# MAIN: read → combine → featurize → save
# =========================
# if __name__ == "__main__":
#     df = load_email_excel(INPUT_PATH) if IS_EXCEL else load_email_csv(INPUT_PATH)

#     # รวม Subject + Body (จะโฟกัสข้อความมากกว่า URL/อีเมล เพราะเราตัดทิ้งแล้ว)
#     combined = df["Subject"].fillna("") + "\n" + df["Body"].fillna("")

#     # ดึงฟีเจอร์ทีละแถวแล้วแปลงเป็น DataFrame
#     features_df = combined.apply(extract_stylo_features).apply(pd.Series)

#     # ต่อคอลัมน์เดิม + ฟีเจอร์ แล้วเซฟ
#     out = pd.concat([df.reset_index(drop=True), features_df], axis=1)
#     out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

#     # แสดงสรุป
#     print(f"บันทึกไฟล์แล้ว → {OUTPUT_CSV}")
#     print(out.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.9 MB/s eta 0:00:00


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [ ]:
# Punctuation Features (2 features)
## 1. Punctuation Frequency
## 2. Punctuation Variety

def punctuation_frequency(row):
  s = str(row["Subject"]) + " " + str(row["Body"])
  s_count = len(s)
  target_punc = ['.', ',', ':', '-', '"', '(', ')', '/', '\\' ]

  total_punc_count = 0
  variety_count = 0

  if (s_count==0):
    return {
        "punctuation_frequency" : 0.0,
   }
  for punc in target_punc:
    punc_count = s.count(punc)
    total_punc_count += punc_count
    if (punc_count>0):
      variety_count += 1
  return {"punctuation_frequenc": total_punc_count/s_count  }

def punctuation_variety(row):
  s = str(row["Subject"]) + " " + str(row["Body"])
  s_count = len(s)
  target_punc = ['.', ',', ':', '-', '"', '(', ')', '/', '\\' ]

  total_punc_count = 0
  variety_count = 0

  if (s_count==0):
    return {"punctuation_variety" : 0 }
  for punc in target_punc:
    punc_count = s.count(punc)
    total_punc_count += punc_count
    if (punc_count>0):
      variety_count += 1
  return {"punctutation_variety" : variety_count}

# Readability Scores Features (5 features)
## 1. Flesch Reading Ease
## 2. SMOG Index
## 3. Dale--Chall Readability Score
## 4. Coleman--Liau Index (CLI)
## 5. Gunning Fog Index (GFI)

def fleasch_reading_ease(row):
  s = str(row["Subject"]) + " " + str(row["Body"])
  return { "flesch_reading_ease" : textstat.flesch_reading_ease(s)}

def smog_index(row):
  s = str(row["Subject"]) + " " + str(row["Body"])
  return { "smog_index" : textstat.smog_index(s) }

def dale_chall_readability_score(row):
  s = str(row["Subject"]) + " " + str(row["Body"])
  return  { "dale_chall_readability_socre" : textstat.dale_chall_readability_score_v2(s) }

def coleman_liau_index(row):
  s = str(row["Subject"]) + " " + str(row["Body"])
  return  { "coleman_liau_index" : textstat.coleman_liau_index(s) }

def gunning_fog_index(row):
  s = str(row["Subject"]) + " " + str(row["Body"])
  return { "gunning_fog_index" : textstat.gunning_fog(s) }